In [1]:
from __future__ import annotations

import asyncio
import json
import random
import time
from datetime import datetime, timedelta, timezone
from uuid import uuid4

import pandas as pd
import vertexai
from google import genai
from google.api_core.exceptions import Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage
from google.genai import types
from vertexai.language_models import TextEmbeddingModel

In [2]:
PROJECT_ID = "leafy-guide-497515-m4"
LOCATION = "us-central1"

BUCKET_NAME = "leafy-guide-497515-m4-vector-assets"

DATASET_ID = "synthetic_marketplace_data_factory"

RUN_TABLE_ID = "generation_runs"
CUSTOMER_TABLE_ID = "synthetic_customers"
PRODUCT_TABLE_ID = "synthetic_products"
ORDER_TABLE_ID = "synthetic_orders"
ORDER_ITEM_TABLE_ID = "synthetic_order_items"
CHUNK_TABLE_ID = "synthetic_product_chunks"
BENCHMARK_TABLE_ID = "batch_load_benchmarks"

PLANNING_MODEL = "gemini-2.5-pro"
TEXT_MODEL = "gemini-2.5-flash"
TEXT_EMBEDDING_MODEL = "text-embedding-005"

CUSTOMER_COUNT = 120
PRODUCT_COUNT = 30
ORDER_COUNT = 400

MAX_CONCURRENT_GEMINI_CALLS = 5
GEMINI_MAX_RETRIES = 3

CHUNK_SIZE_CHARS = 650
CHUNK_OVERLAP_CHARS = 100

EMBEDDING_BATCH_SIZE = 16

GCS_STAGING_PREFIX = "synthetic-data-factory/staging"
GCS_REPORT_PREFIX = "synthetic-data-factory/reports"
GCS_SUMMARY_PREFIX = "synthetic-data-factory/summaries"

RANDOM_SEED = 42

print("Configuration loaded.")
print("Project:", PROJECT_ID)
print("Vertex AI location:", LOCATION)
print("Bucket:", BUCKET_NAME)
print("Customers:", CUSTOMER_COUNT)
print("Products:", PRODUCT_COUNT)
print("Orders:", ORDER_COUNT)
print("Gemini concurrency:", MAX_CONCURRENT_GEMINI_CALLS)

Configuration loaded.
Project: leafy-guide-497515-m4
Vertex AI location: us-central1
Bucket: leafy-guide-497515-m4-vector-assets
Customers: 120
Products: 30
Orders: 400
Gemini concurrency: 5


In [3]:
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

google_vertex_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

async_google_vertex_client = google_vertex_client.aio

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

bucket.reload()

bigquery_client = bigquery.Client(project=PROJECT_ID)

text_embedding_model = TextEmbeddingModel.from_pretrained(
    TEXT_EMBEDDING_MODEL
)

BUCKET_LOCATION = bucket.location

if BUCKET_LOCATION in {"US", "EU"}:
    BIGQUERY_LOCATION = BUCKET_LOCATION
else:
    BIGQUERY_LOCATION = BUCKET_LOCATION.lower()

print("Clients created.")
print("Bucket exists:", bucket.exists())
print("Bucket location:", BUCKET_LOCATION)
print("BigQuery dataset location:", BIGQUERY_LOCATION)
print("BigQuery project:", bigquery_client.project)
print("Text embedding model loaded.")

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Clients created.
Bucket exists: True
Bucket location: EU
BigQuery dataset location: EU
BigQuery project: leafy-guide-497515-m4
Text embedding model loaded.


In [4]:
dataset_ref = bigquery.Dataset(
    f"{PROJECT_ID}.{DATASET_ID}"
)

dataset_ref.location = BIGQUERY_LOCATION

try:
    dataset = bigquery_client.create_dataset(dataset_ref)

    print(
        "Created dataset:",
        dataset.full_dataset_id,
    )

except Conflict:
    dataset = bigquery_client.get_dataset(dataset_ref)

    print(
        "Dataset already exists:",
        dataset.full_dataset_id,
    )

run_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{RUN_TABLE_ID}"
)

customer_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{CUSTOMER_TABLE_ID}"
)

product_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{PRODUCT_TABLE_ID}"
)

order_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{ORDER_TABLE_ID}"
)

order_item_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{ORDER_ITEM_TABLE_ID}"
)

chunk_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{CHUNK_TABLE_ID}"
)

benchmark_table_ref = (
    f"{PROJECT_ID}.{DATASET_ID}.{BENCHMARK_TABLE_ID}"
)

print("Run table:", run_table_ref)
print("Customer table:", customer_table_ref)
print("Product table:", product_table_ref)
print("Order table:", order_table_ref)
print("Order item table:", order_item_table_ref)
print("Chunk table:", chunk_table_ref)
print("Benchmark table:", benchmark_table_ref)

Created dataset: leafy-guide-497515-m4:synthetic_marketplace_data_factory
Run table: leafy-guide-497515-m4.synthetic_marketplace_data_factory.generation_runs
Customer table: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_customers
Product table: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_products
Order table: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_orders
Order item table: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_order_items
Chunk table: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_product_chunks
Benchmark table: leafy-guide-497515-m4.synthetic_marketplace_data_factory.batch_load_benchmarks


In [6]:
def ensure_bigquery_table(
    table_ref: str,
    schema: list[bigquery.SchemaField],
) -> bigquery.Table:
    try:
        table = bigquery_client.get_table(table_ref)

        print(
            "Table already exists:",
            table.full_table_id,
        )

        return table

    except NotFound:
        print(
            "Table does not exist. Creating:",
            table_ref,
        )

        table = bigquery.Table(
            table_ref,
            schema=schema,
        )

        created_table = bigquery_client.create_table(
            table
        )

        print(
            "Created table:",
            created_table.full_table_id,
        )

        return created_table

In [7]:
run_schema = [
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "marketplace_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "customer_count",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "product_count",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "order_count",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "order_item_count",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "chunk_count",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "plan_json",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]


customer_schema = [
    bigquery.SchemaField(
        "customer_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "customer_code",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "email",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "region",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "customer_segment",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "company_size",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "active",
        "BOOLEAN",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]


product_schema = [
    bigquery.SchemaField(
        "product_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "product_number",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "sku",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "category",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "base_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "marketing_title",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "semantic_description",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "use_cases",
        "STRING",
        mode="REPEATED",
    ),
    bigquery.SchemaField(
        "search_keywords",
        "STRING",
        mode="REPEATED",
    ),
    bigquery.SchemaField(
        "unit_price",
        "FLOAT64",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "stock_quantity",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "attributes_json",
        "STRING",
        mode="NULLABLE",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]


order_schema = [
    bigquery.SchemaField(
        "order_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "order_number",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "customer_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "order_status",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "currency",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "total_amount",
        "FLOAT64",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "order_timestamp",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]


order_item_schema = [
    bigquery.SchemaField(
        "order_item_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "order_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "product_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "quantity",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "unit_price",
        "FLOAT64",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "line_total",
        "FLOAT64",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]


chunk_schema = [
    bigquery.SchemaField(
        "chunk_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "product_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "sku",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "category",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "title",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "chunk_number",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "chunk_text",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "chunk_char_count",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "embedding_model",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "embedding",
        "FLOAT64",
        mode="REPEATED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]


benchmark_schema = [
    bigquery.SchemaField(
        "benchmark_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "run_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "table_name",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "gcs_uri",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "row_count",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "source_bytes",
        "INTEGER",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "load_seconds",
        "FLOAT64",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "bigquery_job_id",
        "STRING",
        mode="REQUIRED",
    ),
    bigquery.SchemaField(
        "created_at",
        "TIMESTAMP",
        mode="REQUIRED",
    ),
]

In [8]:
run_table = ensure_bigquery_table(
    run_table_ref,
    run_schema,
)

customer_table = ensure_bigquery_table(
    customer_table_ref,
    customer_schema,
)

product_table = ensure_bigquery_table(
    product_table_ref,
    product_schema,
)

order_table = ensure_bigquery_table(
    order_table_ref,
    order_schema,
)

order_item_table = ensure_bigquery_table(
    order_item_table_ref,
    order_item_schema,
)

chunk_table = ensure_bigquery_table(
    chunk_table_ref,
    chunk_schema,
)

benchmark_table = ensure_bigquery_table(
    benchmark_table_ref,
    benchmark_schema,
)

Table does not exist. Creating: leafy-guide-497515-m4.synthetic_marketplace_data_factory.generation_runs
Created table: leafy-guide-497515-m4:synthetic_marketplace_data_factory.generation_runs
Table does not exist. Creating: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_customers
Created table: leafy-guide-497515-m4:synthetic_marketplace_data_factory.synthetic_customers
Table does not exist. Creating: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_products
Created table: leafy-guide-497515-m4:synthetic_marketplace_data_factory.synthetic_products
Table does not exist. Creating: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_orders
Created table: leafy-guide-497515-m4:synthetic_marketplace_data_factory.synthetic_orders
Table does not exist. Creating: leafy-guide-497515-m4.synthetic_marketplace_data_factory.synthetic_order_items
Created table: leafy-guide-497515-m4:synthetic_marketplace_data_factory.synthetic_order_items
Table d

In [9]:
def gcs_uri_from_blob_name(
    bucket_name: str,
    blob_name: str,
) -> str:
    return f"gs://{bucket_name}/{blob_name}"


def rows_to_ndjson(
    rows: list[dict],
) -> str:
    return "\n".join(
        json.dumps(
            row,
            ensure_ascii=False,
            default=str,
        )
        for row in rows
    )


def upload_text_to_gcs(
    text: str,
    *,
    blob_name: str,
    content_type: str,
) -> tuple[str, int]:
    data = text.encode("utf-8")

    blob = bucket.blob(blob_name)

    blob.upload_from_string(
        data,
        content_type=content_type,
    )

    gcs_uri = gcs_uri_from_blob_name(
        bucket_name=BUCKET_NAME,
        blob_name=blob_name,
    )

    return gcs_uri, len(data)


def upload_rows_as_ndjson(
    rows: list[dict],
    *,
    blob_name: str,
) -> tuple[str, int]:
    ndjson_text = rows_to_ndjson(rows)

    return upload_text_to_gcs(
        ndjson_text,
        blob_name=blob_name,
        content_type="application/x-ndjson",
    )

In [10]:
DATASET_PLAN_RESPONSE_SCHEMA = {
    "required": [
        "marketplace_name",
        "marketplace_description",
        "regions",
        "customer_segments",
        "company_sizes",
        "product_categories",
        "order_statuses",
        "business_questions",
    ],
    "properties": {
        "marketplace_name": {
            "type": "STRING",
        },
        "marketplace_description": {
            "type": "STRING",
        },
        "regions": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "customer_segments": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "company_sizes": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "product_categories": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "order_statuses": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "business_questions": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
    },
    "type": "OBJECT",
}

print("Dataset plan response schema created.")

Dataset plan response schema created.


In [11]:
def generate_marketplace_plan() -> dict:
    prompt = """
Design a synthetic B2B marketplace dataset.

The marketplace sells equipment for field operations,
industrial maintenance, infrastructure teams, and technical engineers.

Create a coherent synthetic business domain.

Requirements:
- exactly 6 regions
- exactly 5 customer segments
- exactly 4 company sizes
- exactly 6 product categories
- exactly 5 order statuses
- exactly 6 analytical business questions
- category names should be suitable for technical and industrial equipment
- all data must be fictional
"""

    response = google_vertex_client.models.generate_content(
        model=PLANNING_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=DATASET_PLAN_RESPONSE_SCHEMA,
            temperature=0.3,
        ),
    )

    return json.loads(response.text)


marketplace_plan = generate_marketplace_plan()

print(
    json.dumps(
        marketplace_plan,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "marketplace_name": "InfraTech Exchange",
  "marketplace_description": "A B2B marketplace specializing in high-performance equipment and tools for field operations, industrial maintenance, infrastructure projects, and technical engineering professionals.",
  "regions": [
    "North America",
    "South America",
    "EMEA (Europe, Middle East, Africa)",
    "APAC (Asia-Pacific)",
    "Central Asia",
    "Oceania"
  ],
  "customer_segments": [
    "Energy & Utilities",
    "Telecommunications",
    "Civil Engineering & Construction",
    "Manufacturing & Processing",
    "Public Sector Services"
  ],
  "company_sizes": [
    "Small Business (1-50 employees)",
    "Medium Business (51-500 employees)",
    "Large Enterprise (501-5000 employees)",
    "Corporate Conglomerate (5001+ employees)"
  ],
  "product_categories": [
    "Test & Measurement Instruments",
    "Safety & Personal Protective Equipment (PPE)",
    "Heavy Machinery & Lifting Gear",
    "Power Tools & Welding Supplies"

In [12]:
random.seed(RANDOM_SEED)

run_id = str(uuid4())

run_timestamp = datetime.now(timezone.utc)

print("Run ID:", run_id)
print("Run timestamp:", run_timestamp.isoformat())
print("Marketplace:", marketplace_plan["marketplace_name"])

Run ID: 59eda7f6-2bce-4ecc-b3be-e93d6137c658
Run timestamp: 2026-07-06T14:48:23.043116+00:00
Marketplace: InfraTech Exchange


In [13]:
customer_rows = []

regions = marketplace_plan["regions"]
customer_segments = marketplace_plan["customer_segments"]
company_sizes = marketplace_plan["company_sizes"]

for customer_number in range(
    1,
    CUSTOMER_COUNT + 1,
):
    customer_id = str(uuid4())

    customer_rows.append(
        {
            "customer_id": customer_id,
            "customer_code": f"CUST-{customer_number:05d}",
            "email": (
                f"customer_{customer_number:05d}"
                "@synthetic.example.test"
            ),
            "region": random.choice(regions),
            "customer_segment": random.choice(
                customer_segments
            ),
            "company_size": random.choice(
                company_sizes
            ),
            "active": random.random() > 0.08,
            "created_at": (
                run_timestamp
                - timedelta(
                    days=random.randint(10, 900)
                )
            ).isoformat(),
        }
    )

print("Generated customers:", len(customer_rows))

customers_df = pd.DataFrame(customer_rows)

customers_df.head()

Generated customers: 120


,customer_id,customer_code,email,region,customer_segment,company_size,active,created_at
0,7383ea57-acfd-40e1-87c5-38695c9d431f,CUST-00001,customer_00001@synthetic.example.test,Oceania,Energy & Utilities,Small Business (1-50 employees),True,2025-10-19T14:48:23.043116+00:00
1,58e9a193-2dab-4909-8bf0-522f84b14b0f,CUST-00002,customer_00002@synthetic.example.test,South America,Telecommunications,Small Business (1-50 employees),True,2024-12-15T14:48:23.043116+00:00
2,8a706ca1-60d0-4f5c-aaa9-bc0d0111fed0,CUST-00003,customer_00003@synthetic.example.test,North America,Public Sector Services,Corporate Conglomerate (5001+ employees),False,2026-03-23T14:48:23.043116+00:00
3,157c2ad5-f820-4d4f-b751-e66841d6de8f,CUST-00004,customer_00004@synthetic.example.test,South America,Telecommunications,Small Business (1-50 employees),True,2024-06-23T14:48:23.043116+00:00
4,446b976a-5ed3-498f-90c9-47140d71c559,CUST-00005,customer_00005@synthetic.example.test,Oceania,Public Sector Services,Corporate Conglomerate (5001+ employees),True,2024-10-31T14:48:23.043116+00:00


In [14]:
product_categories = marketplace_plan[
    "product_categories"
]

base_product_rows = []

for product_number in range(
    1,
    PRODUCT_COUNT + 1,
):
    category = product_categories[
        (product_number - 1)
        % len(product_categories)
    ]

    unit_price = round(
        random.uniform(25.0, 750.0),
        2,
    )

    stock_quantity = random.randint(
        0,
        300,
    )

    base_product_rows.append(
        {
            "product_id": str(uuid4()),
            "product_number": product_number,
            "sku": f"FIELD-{product_number:04d}",
            "category": category,
            "base_name": (
                f"{category.title()} Field Unit "
                f"{product_number:02d}"
            ),
            "unit_price": unit_price,
            "stock_quantity": stock_quantity,
        }
    )

print(
    "Generated base products:",
    len(base_product_rows),
)

pd.DataFrame(base_product_rows).head()

Generated base products: 30


,product_id,product_number,sku,category,base_name,unit_price,stock_quantity
0,550c4873-fa60-4f2d-87a8-49a36b836a83,1,FIELD-0001,Test & Measurement Instruments,Test & Measurement Instruments Field Unit 01,442.38,244
1,0e362757-f209-42ae-b7e4-6537f3c0a630,2,FIELD-0002,Safety & Personal Protective Equipment (PPE),Safety & Personal Protective Equipment (Ppe) F...,29.27,180
2,a4d1119e-76af-492e-bed5-e3c44b4fcd02,3,FIELD-0003,Heavy Machinery & Lifting Gear,Heavy Machinery & Lifting Gear Field Unit 03,241.49,199
3,45895b6e-2e8c-47c8-afc9-c7394415ee5f,4,FIELD-0004,Power Tools & Welding Supplies,Power Tools & Welding Supplies Field Unit 04,643.75,214
4,cbc15218-5be5-439c-9319-3f8ab3800dc3,5,FIELD-0005,Diagnostic & Imaging Systems,Diagnostic & Imaging Systems Field Unit 05,415.21,279


In [15]:
PRODUCT_ENRICHMENT_RESPONSE_SCHEMA = {
    "required": [
        "marketing_title",
        "semantic_description",
        "use_cases",
        "search_keywords",
        "attributes",
    ],
    "properties": {
        "marketing_title": {
            "type": "STRING",
        },
        "semantic_description": {
            "type": "STRING",
        },
        "use_cases": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "search_keywords": {
            "type": "ARRAY",
            "items": {
                "type": "STRING",
            },
        },
        "attributes": {
            "type": "OBJECT",
            "properties": {
                "environment": {
                    "type": "STRING",
                },
                "portability": {
                    "type": "STRING",
                },
                "durability": {
                    "type": "STRING",
                },
                "primary_user": {
                    "type": "STRING",
                },
            },
            "required": [
                "environment",
                "portability",
                "durability",
                "primary_user",
            ],
        },
    },
    "type": "OBJECT",
}

print("Product enrichment schema created.")

Product enrichment schema created.


In [16]:
async def enrich_product_async(
    product: dict,
    semaphore: asyncio.Semaphore,
) -> dict:
    prompt = f"""
You are a B2B industrial product catalog specialist.

Marketplace:
{marketplace_plan["marketplace_name"]}

Marketplace description:
{marketplace_plan["marketplace_description"]}

Product:
{json.dumps(product, indent=2, ensure_ascii=False)}

Create realistic synthetic semantic product metadata.

Requirements:
- marketing_title should be concise
- semantic_description should contain 180 to 320 words
- include operational capabilities
- include suitable environments
- include problems the product solves
- include likely technical users
- exactly 4 use cases
- exactly 8 search keywords
- no real brands
- no real product model names
"""

    async with semaphore:
        for attempt in range(
            1,
            GEMINI_MAX_RETRIES + 1,
        ):
            try:
                response = (
                    await async_google_vertex_client
                    .models
                    .generate_content(
                        model=TEXT_MODEL,
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            response_mime_type=(
                                "application/json"
                            ),
                            response_schema=(
                                PRODUCT_ENRICHMENT_RESPONSE_SCHEMA
                            ),
                            temperature=0.4,
                        ),
                    )
                )

                enrichment = json.loads(
                    response.text
                )

                return {
                    **product,
                    "marketing_title": (
                        enrichment["marketing_title"]
                    ),
                    "semantic_description": (
                        enrichment[
                            "semantic_description"
                        ]
                    ),
                    "use_cases": enrichment["use_cases"],
                    "search_keywords": enrichment[
                        "search_keywords"
                    ],
                    "attributes_json": json.dumps(
                        enrichment["attributes"],
                        ensure_ascii=False,
                    ),
                    "created_at": (
                        datetime.now(timezone.utc)
                        .isoformat()
                    ),
                }

            except Exception as exc:
                print(
                    "Attempt failed:",
                    attempt,
                    "| product:",
                    product["sku"],
                    "| error:",
                    repr(exc),
                )

                if attempt >= GEMINI_MAX_RETRIES:
                    raise

                await asyncio.sleep(
                    2 ** attempt
                )

    raise RuntimeError(
        f"Product enrichment failed: {product['sku']}"
    )

In [17]:
async def enrich_all_products_async(
    products: list[dict],
) -> list[dict]:
    semaphore = asyncio.Semaphore(
        MAX_CONCURRENT_GEMINI_CALLS
    )

    tasks = [
        enrich_product_async(
            product,
            semaphore,
        )
        for product in products
    ]

    return await asyncio.gather(*tasks)


enrichment_started_at = time.perf_counter()

product_rows = await enrich_all_products_async(
    base_product_rows
)

enrichment_seconds = (
    time.perf_counter()
    - enrichment_started_at
)

print("Enriched products:", len(product_rows))
print(
    "Async enrichment seconds:",
    round(enrichment_seconds, 2),
)

products_df = pd.DataFrame(
    [
        {
            "sku": product["sku"],
            "category": product["category"],
            "marketing_title": product[
                "marketing_title"
            ],
            "unit_price": product["unit_price"],
            "stock_quantity": product[
                "stock_quantity"
            ],
        }
        for product in product_rows
    ]
)

products_df.head()

Attempt failed: 1 | product: FIELD-0008 | error: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}")
Attempt failed: 1 | product: FIELD-0012 | error: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}")
Attempt failed: 2 | product: FIELD-0008 | error: ClientError("429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}")
Attempt failed: 1 | product: FIELD-0030 | error: ClientError("429 RESOURCE_EXHAUST

,sku,category,marketing_title,unit_price,stock_quantity
0,FIELD-0001,Test & Measurement Instruments,Rugged Industrial Field Measurement & Diagnost...,442.38,244
1,FIELD-0002,Safety & Personal Protective Equipment (PPE),Essential Field Operations PPE Kit,29.27,180
2,FIELD-0003,Heavy Machinery & Lifting Gear,Rugged Field-Deployable Heavy Lifting Unit,241.49,199
3,FIELD-0004,Power Tools & Welding Supplies,Industrial Field Power & Welding System,643.75,214
4,FIELD-0005,Diagnostic & Imaging Systems,Rugged Field Diagnostic & Imaging System,415.21,279


In [18]:
example_product = product_rows[0]

print("SKU:")
print(example_product["sku"])

print("\nCategory:")
print(example_product["category"])

print("\nMarketing title:")
print(example_product["marketing_title"])

print("\nSemantic description:")
print(example_product["semantic_description"])

print("\nUse cases:")
print(example_product["use_cases"])

print("\nSearch keywords:")
print(example_product["search_keywords"])

print("\nAttributes:")
print(example_product["attributes_json"])

SKU:
FIELD-0001

Category:
Test & Measurement Instruments

Marketing title:
Rugged Industrial Field Measurement & Diagnostic Unit

Semantic description:
The Rugged Industrial Field Measurement & Diagnostic Unit is an essential tool engineered for precision and reliability in demanding operational environments. This advanced portable instrument provides comprehensive multi-parameter measurement capabilities, including accurate readings for voltage, current, resistance, frequency, temperature, pressure, and vibration. Equipped with real-time data analysis and robust data logging features, it empowers technical professionals to conduct thorough diagnostics, monitor system performance, and identify anomalies swiftly. Its intuitive user interface and wireless connectivity options streamline data collection and reporting, enhancing efficiency in the field. Designed to withstand the rigors of harsh industrial settings, remote infrastructure sites, and challenging outdoor conditions, this unit

In [19]:
order_rows = []
order_item_rows = []

order_statuses = marketplace_plan[
    "order_statuses"
]

for order_number in range(
    1,
    ORDER_COUNT + 1,
):
    customer = random.choice(customer_rows)

    order_id = str(uuid4())

    order_timestamp = (
        run_timestamp
        - timedelta(
            days=random.randint(0, 180),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
        )
    )

    item_count = random.randint(1, 5)

    selected_products = random.sample(
        product_rows,
        k=item_count,
    )

    order_total = 0.0

    for product in selected_products:
        quantity = random.randint(1, 6)

        unit_price = product["unit_price"]

        line_total = round(
            quantity * unit_price,
            2,
        )

        order_total += line_total

        order_item_rows.append(
            {
                "order_item_id": str(uuid4()),
                "order_id": order_id,
                "product_id": product["product_id"],
                "quantity": quantity,
                "unit_price": unit_price,
                "line_total": line_total,
                "created_at": (
                    run_timestamp.isoformat()
                ),
            }
        )

    order_rows.append(
        {
            "order_id": order_id,
            "order_number": order_number,
            "customer_id": customer["customer_id"],
            "order_status": random.choice(
                order_statuses
            ),
            "currency": "USD",
            "total_amount": round(
                order_total,
                2,
            ),
            "order_timestamp": (
                order_timestamp.isoformat()
            ),
            "created_at": (
                run_timestamp.isoformat()
            ),
        }
    )

print("Generated orders:", len(order_rows))
print(
    "Generated order items:",
    len(order_item_rows),
)

pd.DataFrame(order_rows).head()

Generated orders: 400
Generated order items: 1193


,order_id,order_number,customer_id,order_status,currency,total_amount,order_timestamp,created_at
0,1edc788f-a7b7-4d0b-8169-22ffafd7a626,1,9aa0ffa2-2fd1-468f-81a0-1d06b1d0863e,Shipped,USD,4231.26,2026-01-08T05:59:23.043116+00:00,2026-07-06T14:48:23.043116+00:00
1,8b5dff35-7da4-40d8-a240-52897ac50203,2,239cdcce-0106-439b-a72b-34a2bc91cf29,Processing,USD,6374.72,2026-02-08T13:49:23.043116+00:00,2026-07-06T14:48:23.043116+00:00
2,4fb50846-f898-4f79-bbcf-4c2efaf5ec01,3,8c0f47e1-1fc1-4aa8-b9e4-3a286c96199c,Cancelled,USD,2307.81,2026-02-04T15:55:23.043116+00:00,2026-07-06T14:48:23.043116+00:00
3,76a7b22b-4228-4399-b1d3-6b873dbb43e5,4,0cd18988-9ba5-4e83-900d-956a9bae1c3f,Delivered,USD,222.75,2026-06-05T19:58:23.043116+00:00,2026-07-06T14:48:23.043116+00:00
4,2c7a7800-8a17-46e5-8290-16c6bc16f359,5,d3b5b702-df76-471c-adb6-4e7e5a36a9e4,Shipped,USD,2588.43,2026-04-03T12:16:23.043116+00:00,2026-07-06T14:48:23.043116+00:00


In [20]:
customer_ids = {
    row["customer_id"]
    for row in customer_rows
}

product_ids = {
    row["product_id"]
    for row in product_rows
}

order_ids = {
    row["order_id"]
    for row in order_rows
}

invalid_order_customers = [
    row
    for row in order_rows
    if row["customer_id"] not in customer_ids
]

invalid_item_orders = [
    row
    for row in order_item_rows
    if row["order_id"] not in order_ids
]

invalid_item_products = [
    row
    for row in order_item_rows
    if row["product_id"] not in product_ids
]

order_total_from_items = {}

for item in order_item_rows:
    order_total_from_items.setdefault(
        item["order_id"],
        0.0,
    )

    order_total_from_items[
        item["order_id"]
    ] += item["line_total"]


order_total_mismatches = []

for order in order_rows:
    calculated_total = round(
        order_total_from_items[
            order["order_id"]
        ],
        2,
    )

    if calculated_total != order["total_amount"]:
        order_total_mismatches.append(
            {
                "order_id": order["order_id"],
                "stored_total": order[
                    "total_amount"
                ],
                "calculated_total": (
                    calculated_total
                ),
            }
        )


print(
    "Invalid order customer refs:",
    len(invalid_order_customers),
)

print(
    "Invalid item order refs:",
    len(invalid_item_orders),
)

print(
    "Invalid item product refs:",
    len(invalid_item_products),
)

print(
    "Order total mismatches:",
    len(order_total_mismatches),
)

assert not invalid_order_customers
assert not invalid_item_orders
assert not invalid_item_products
assert not order_total_mismatches

print("Relational integrity validation passed.")

Invalid order customer refs: 0
Invalid item order refs: 0
Invalid item product refs: 0
Order total mismatches: 0
Relational integrity validation passed.


In [21]:
def normalize_text(text: str) -> str:
    return " ".join(
        text.split()
    )


def split_text_into_chunks(
    text: str,
    *,
    chunk_size_chars: int = CHUNK_SIZE_CHARS,
    chunk_overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> list[str]:
    clean_text = normalize_text(text)

    if len(clean_text) <= chunk_size_chars:
        return [clean_text]

    chunks = []

    start = 0

    while start < len(clean_text):
        end = min(
            start + chunk_size_chars,
            len(clean_text),
        )

        if end < len(clean_text):
            sentence_boundary = clean_text.rfind(
                ". ",
                start,
                end,
            )

            if sentence_boundary > (
                start
                + int(chunk_size_chars * 0.6)
            ):
                end = sentence_boundary + 1

        chunk = clean_text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(clean_text):
            break

        start = max(
            0,
            end - chunk_overlap_chars,
        )

    return chunks

In [22]:
def build_product_semantic_document(
    product: dict,
) -> str:
    return "\n".join(
        [
            (
                f"Product title: "
                f"{product['marketing_title']}."
            ),
            (
                f"SKU: "
                f"{product['sku']}."
            ),
            (
                f"Category: "
                f"{product['category']}."
            ),
            (
                f"Description: "
                f"{product['semantic_description']}"
            ),
            (
                "Use cases: "
                + "; ".join(
                    product["use_cases"]
                )
                + "."
            ),
            (
                "Search concepts: "
                + ", ".join(
                    product["search_keywords"]
                )
                + "."
            ),
            (
                "Technical attributes: "
                f"{product['attributes_json']}."
            ),
            (
                f"Unit price: "
                f"{product['unit_price']} USD."
            ),
            (
                f"Available stock: "
                f"{product['stock_quantity']} units."
            ),
        ]
    )


product_semantic_documents = {
    product["product_id"]: (
        build_product_semantic_document(
            product
        )
    )
    for product in product_rows
}

first_product_id = product_rows[0]["product_id"]

print(
    product_semantic_documents[
        first_product_id
    ]
)

Product title: Rugged Industrial Field Measurement & Diagnostic Unit.
SKU: FIELD-0001.
Category: Test & Measurement Instruments.
Description: The Rugged Industrial Field Measurement & Diagnostic Unit is an essential tool engineered for precision and reliability in demanding operational environments. This advanced portable instrument provides comprehensive multi-parameter measurement capabilities, including accurate readings for voltage, current, resistance, frequency, temperature, pressure, and vibration. Equipped with real-time data analysis and robust data logging features, it empowers technical professionals to conduct thorough diagnostics, monitor system performance, and identify anomalies swiftly. Its intuitive user interface and wireless connectivity options streamline data collection and reporting, enhancing efficiency in the field. Designed to withstand the rigors of harsh industrial settings, remote infrastructure sites, and challenging outdoor conditions, this unit ensures co

In [23]:
chunk_rows_without_embeddings = []

for product in product_rows:
    semantic_document = (
        product_semantic_documents[
            product["product_id"]
        ]
    )

    product_chunks = split_text_into_chunks(
        semantic_document
    )

    print(
        "Product:",
        product["sku"],
        "| chunks:",
        len(product_chunks),
    )

    for chunk_number, chunk_text in enumerate(
        product_chunks,
        start=1,
    ):
        chunk_rows_without_embeddings.append(
            {
                "chunk_id": str(uuid4()),
                "product_id": product["product_id"],
                "sku": product["sku"],
                "category": product["category"],
                "title": product["marketing_title"],
                "chunk_number": chunk_number,
                "chunk_text": chunk_text,
                "chunk_char_count": len(chunk_text),
                "embedding_model": (
                    TEXT_EMBEDDING_MODEL
                ),
                "created_at": (
                    run_timestamp.isoformat()
                ),
            }
        )

print(
    "Product chunks:",
    len(chunk_rows_without_embeddings),
)

chunks_preview_df = pd.DataFrame(
    [
        {
            "sku": row["sku"],
            "category": row["category"],
            "chunk_number": row["chunk_number"],
            "chunk_char_count": row[
                "chunk_char_count"
            ],
            "preview": row["chunk_text"][:180],
        }
        for row in chunk_rows_without_embeddings
    ]
)

chunks_preview_df.head(10)

Product: FIELD-0001 | chunks: 5
Product: FIELD-0002 | chunks: 6
Product: FIELD-0003 | chunks: 5
Product: FIELD-0004 | chunks: 5
Product: FIELD-0005 | chunks: 6
Product: FIELD-0006 | chunks: 6
Product: FIELD-0007 | chunks: 5
Product: FIELD-0008 | chunks: 6
Product: FIELD-0009 | chunks: 5
Product: FIELD-0010 | chunks: 6
Product: FIELD-0011 | chunks: 6
Product: FIELD-0012 | chunks: 6
Product: FIELD-0013 | chunks: 5
Product: FIELD-0014 | chunks: 6
Product: FIELD-0015 | chunks: 6
Product: FIELD-0016 | chunks: 6
Product: FIELD-0017 | chunks: 6
Product: FIELD-0018 | chunks: 6
Product: FIELD-0019 | chunks: 6
Product: FIELD-0020 | chunks: 7
Product: FIELD-0021 | chunks: 6
Product: FIELD-0022 | chunks: 6
Product: FIELD-0023 | chunks: 6
Product: FIELD-0024 | chunks: 6
Product: FIELD-0025 | chunks: 6
Product: FIELD-0026 | chunks: 7
Product: FIELD-0027 | chunks: 6
Product: FIELD-0028 | chunks: 5
Product: FIELD-0029 | chunks: 5
Product: FIELD-0030 | chunks: 6
Product chunks: 174


,sku,category,chunk_number,chunk_char_count,preview
0,FIELD-0001,Test & Measurement Instruments,1,510,Product title: Rugged Industrial Field Measure...
1,FIELD-0001,Test & Measurement Instruments,2,630,"accurate readings for voltage, current, resist..."
2,FIELD-0001,Test & Measurement Instruments,3,612,"es, and challenging outdoor conditions, this u..."
3,FIELD-0001,Test & Measurement Instruments,4,568,necessary for informed decision-making across ...
4,FIELD-0001,Test & Measurement Instruments,5,393,"ool, Infrastructure Monitoring, Sensor Calibra..."
5,FIELD-0002,Safety & Personal Protective Equipment (PPE),1,633,Product title: Essential Field Operations PPE ...
6,FIELD-0002,Safety & Personal Protective Equipment (PPE),2,464,"proactive risk mitigation, ensuring personnel ..."
7,FIELD-0002,Safety & Personal Protective Equipment (PPE),3,467,"nufacturing facilities, remote infrastructure ..."
8,FIELD-0002,Safety & Personal Protective Equipment (PPE),4,636,", making it an ideal choice for organizations ..."
9,FIELD-0002,Safety & Personal Protective Equipment (PPE),5,647,scenarios. Its intuitive design ensures rapid ...


In [24]:
def get_text_embeddings_in_batches(
    texts: list[str],
    *,
    batch_size: int = EMBEDDING_BATCH_SIZE,
) -> list[list[float]]:
    all_embeddings = []

    for start_index in range(
        0,
        len(texts),
        batch_size,
    ):
        end_index = min(
            start_index + batch_size,
            len(texts),
        )

        batch = texts[
            start_index:end_index
        ]

        print(
            "Embedding batch:",
            start_index,
            "->",
            end_index - 1,
            "| size:",
            len(batch),
        )

        batch_embeddings = (
            text_embedding_model
            .get_embeddings(batch)
        )

        all_embeddings.extend(
            [
                list(embedding.values)
                for embedding
                in batch_embeddings
            ]
        )

    return all_embeddings

In [25]:
chunk_texts = [
    row["chunk_text"]
    for row in chunk_rows_without_embeddings
]

embedding_started_at = time.perf_counter()

chunk_embeddings = get_text_embeddings_in_batches(
    chunk_texts
)

embedding_seconds = (
    time.perf_counter()
    - embedding_started_at
)

assert (
    len(chunk_embeddings)
    == len(chunk_rows_without_embeddings)
)

chunk_rows = []

for row, embedding in zip(
    chunk_rows_without_embeddings,
    chunk_embeddings,
    strict=True,
):
    chunk_rows.append(
        {
            **row,
            "embedding": embedding,
        }
    )

print(
    "Chunks with embeddings:",
    len(chunk_rows),
)

print(
    "Embedding seconds:",
    round(embedding_seconds, 2),
)

print(
    "Embedding dimension:",
    len(chunk_rows[0]["embedding"]),
)

Embedding batch: 0 -> 15 | size: 16
Embedding batch: 16 -> 31 | size: 16
Embedding batch: 32 -> 47 | size: 16
Embedding batch: 48 -> 63 | size: 16
Embedding batch: 64 -> 79 | size: 16
Embedding batch: 80 -> 95 | size: 16
Embedding batch: 96 -> 111 | size: 16
Embedding batch: 112 -> 127 | size: 16
Embedding batch: 128 -> 143 | size: 16
Embedding batch: 144 -> 159 | size: 16
Embedding batch: 160 -> 173 | size: 14
Chunks with embeddings: 174
Embedding seconds: 7.53
Embedding dimension: 768


In [26]:
generation_run_row = {
    "run_id": run_id,
    "marketplace_name": marketplace_plan[
        "marketplace_name"
    ],
    "customer_count": len(customer_rows),
    "product_count": len(product_rows),
    "order_count": len(order_rows),
    "order_item_count": len(order_item_rows),
    "chunk_count": len(chunk_rows),
    "plan_json": json.dumps(
        marketplace_plan,
        ensure_ascii=False,
        default=str,
    ),
    "created_at": run_timestamp.isoformat(),
}

print(
    json.dumps(
        generation_run_row,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "run_id": "59eda7f6-2bce-4ecc-b3be-e93d6137c658",
  "marketplace_name": "InfraTech Exchange",
  "customer_count": 120,
  "product_count": 30,
  "order_count": 400,
  "order_item_count": 1193,
  "chunk_count": 174,
  "plan_json": "{\"marketplace_name\": \"InfraTech Exchange\", \"marketplace_description\": \"A B2B marketplace specializing in high-performance equipment and tools for field operations, industrial maintenance, infrastructure projects, and technical engineering professionals.\", \"regions\": [\"North America\", \"South America\", \"EMEA (Europe, Middle East, Africa)\", \"APAC (Asia-Pacific)\", \"Central Asia\", \"Oceania\"], \"customer_segments\": [\"Energy & Utilities\", \"Telecommunications\", \"Civil Engineering & Construction\", \"Manufacturing & Processing\", \"Public Sector Services\"], \"company_sizes\": [\"Small Business (1-50 employees)\", \"Medium Business (51-500 employees)\", \"Large Enterprise (501-5000 employees)\", \"Corporate Conglomerate (5001+ employees)

In [27]:
table_payloads = {
    "generation_runs": {
        "rows": [generation_run_row],
        "table_ref": run_table_ref,
        "schema": run_schema,
    },
    "synthetic_customers": {
        "rows": customer_rows,
        "table_ref": customer_table_ref,
        "schema": customer_schema,
    },
    "synthetic_products": {
        "rows": product_rows,
        "table_ref": product_table_ref,
        "schema": product_schema,
    },
    "synthetic_orders": {
        "rows": order_rows,
        "table_ref": order_table_ref,
        "schema": order_schema,
    },
    "synthetic_order_items": {
        "rows": order_item_rows,
        "table_ref": order_item_table_ref,
        "schema": order_item_schema,
    },
    "synthetic_product_chunks": {
        "rows": chunk_rows,
        "table_ref": chunk_table_ref,
        "schema": chunk_schema,
    },
}

for table_name, payload in table_payloads.items():
    print(
        table_name,
        "rows:",
        len(payload["rows"]),
    )

generation_runs rows: 1
synthetic_customers rows: 120
synthetic_products rows: 30
synthetic_orders rows: 400
synthetic_order_items rows: 1193
synthetic_product_chunks rows: 174


In [28]:
generation_run_row = {
    "run_id": run_id,
    "marketplace_name": marketplace_plan[
        "marketplace_name"
    ],
    "customer_count": len(customer_rows),
    "product_count": len(product_rows),
    "order_count": len(order_rows),
    "order_item_count": len(order_item_rows),
    "chunk_count": len(chunk_rows),
    "plan_json": json.dumps(
        marketplace_plan,
        ensure_ascii=False,
        default=str,
    ),
    "created_at": run_timestamp.isoformat(),
}

print(
    json.dumps(
        generation_run_row,
        indent=2,
        ensure_ascii=False,
    )
)

{
  "run_id": "59eda7f6-2bce-4ecc-b3be-e93d6137c658",
  "marketplace_name": "InfraTech Exchange",
  "customer_count": 120,
  "product_count": 30,
  "order_count": 400,
  "order_item_count": 1193,
  "chunk_count": 174,
  "plan_json": "{\"marketplace_name\": \"InfraTech Exchange\", \"marketplace_description\": \"A B2B marketplace specializing in high-performance equipment and tools for field operations, industrial maintenance, infrastructure projects, and technical engineering professionals.\", \"regions\": [\"North America\", \"South America\", \"EMEA (Europe, Middle East, Africa)\", \"APAC (Asia-Pacific)\", \"Central Asia\", \"Oceania\"], \"customer_segments\": [\"Energy & Utilities\", \"Telecommunications\", \"Civil Engineering & Construction\", \"Manufacturing & Processing\", \"Public Sector Services\"], \"company_sizes\": [\"Small Business (1-50 employees)\", \"Medium Business (51-500 employees)\", \"Large Enterprise (501-5000 employees)\", \"Corporate Conglomerate (5001+ employees)

In [29]:
table_payloads = {
    "generation_runs": {
        "rows": [generation_run_row],
        "table_ref": run_table_ref,
        "schema": run_schema,
    },
    "synthetic_customers": {
        "rows": customer_rows,
        "table_ref": customer_table_ref,
        "schema": customer_schema,
    },
    "synthetic_products": {
        "rows": product_rows,
        "table_ref": product_table_ref,
        "schema": product_schema,
    },
    "synthetic_orders": {
        "rows": order_rows,
        "table_ref": order_table_ref,
        "schema": order_schema,
    },
    "synthetic_order_items": {
        "rows": order_item_rows,
        "table_ref": order_item_table_ref,
        "schema": order_item_schema,
    },
    "synthetic_product_chunks": {
        "rows": chunk_rows,
        "table_ref": chunk_table_ref,
        "schema": chunk_schema,
    },
}

for table_name, payload in table_payloads.items():
    print(
        table_name,
        "rows:",
        len(payload["rows"]),
    )

generation_runs rows: 1
synthetic_customers rows: 120
synthetic_products rows: 30
synthetic_orders rows: 400
synthetic_order_items rows: 1193
synthetic_product_chunks rows: 174


In [30]:
staged_tables = {}

for table_name, payload in table_payloads.items():
    blob_name = (
        f"{GCS_STAGING_PREFIX}/"
        f"{run_id}/"
        f"{table_name}.ndjson"
    )

    gcs_uri, source_bytes = (
        upload_rows_as_ndjson(
            payload["rows"],
            blob_name=blob_name,
        )
    )

    staged_tables[table_name] = {
        **payload,
        "gcs_uri": gcs_uri,
        "source_bytes": source_bytes,
    }

    print("=" * 100)
    print("Table:", table_name)
    print("Rows:", len(payload["rows"]))
    print("Bytes:", source_bytes)
    print("GCS URI:", gcs_uri)

print(
    "Staged tables:",
    len(staged_tables),
)

Table: generation_runs
Rows: 1
Bytes: 2017
GCS URI: gs://leafy-guide-497515-m4-vector-assets/synthetic-data-factory/staging/59eda7f6-2bce-4ecc-b3be-e93d6137c658/generation_runs.ndjson
Table: synthetic_customers
Rows: 120
Bytes: 40204
GCS URI: gs://leafy-guide-497515-m4-vector-assets/synthetic-data-factory/staging/59eda7f6-2bce-4ecc-b3be-e93d6137c658/synthetic_customers.ndjson
Table: synthetic_products
Rows: 30
Bytes: 85589
GCS URI: gs://leafy-guide-497515-m4-vector-assets/synthetic-data-factory/staging/59eda7f6-2bce-4ecc-b3be-e93d6137c658/synthetic_products.ndjson
Table: synthetic_orders
Rows: 400
Bytes: 123091
GCS URI: gs://leafy-guide-497515-m4-vector-assets/synthetic-data-factory/staging/59eda7f6-2bce-4ecc-b3be-e93d6137c658/synthetic_orders.ndjson
Table: synthetic_order_items
Rows: 1193
Bytes: 325703
GCS URI: gs://leafy-guide-497515-m4-vector-assets/synthetic-data-factory/staging/59eda7f6-2bce-4ecc-b3be-e93d6137c658/synthetic_order_items.ndjson
Table: synthetic_product_chunks
Rows: 

In [31]:
def batch_load_ndjson_from_gcs(
    *,
    table_name: str,
    gcs_uri: str,
    table_ref: str,
    schema: list[bigquery.SchemaField],
    expected_row_count: int,
    source_bytes: int,
) -> dict:
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        source_format=(
            bigquery.SourceFormat
            .NEWLINE_DELIMITED_JSON
        ),
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )

    started_at = time.perf_counter()

    load_job = bigquery_client.load_table_from_uri(
        gcs_uri,
        table_ref,
        location=dataset.location,
        job_config=job_config,
    )

    print(
        "Started load job:",
        load_job.job_id,
        "| table:",
        table_name,
    )

    load_job.result()

    load_seconds = (
        time.perf_counter()
        - started_at
    )

    destination_table = (
        bigquery_client.get_table(
            table_ref
        )
    )

    actual_row_count = (
        destination_table.num_rows
    )

    if actual_row_count != expected_row_count:
        raise ValueError(
            f"Row count mismatch for {table_name}: "
            f"expected {expected_row_count}, "
            f"loaded {actual_row_count}"
        )

    return {
        "benchmark_id": str(uuid4()),
        "run_id": run_id,
        "table_name": table_name,
        "gcs_uri": gcs_uri,
        "row_count": actual_row_count,
        "source_bytes": source_bytes,
        "load_seconds": round(
            load_seconds,
            4,
        ),
        "bigquery_job_id": load_job.job_id,
        "created_at": (
            datetime.now(timezone.utc)
            .isoformat()
        ),
    }

In [32]:
load_benchmark_rows = []

for table_name, staged_table in (
    staged_tables.items()
):
    print("\n" + "=" * 100)
    print(
        "Batch loading:",
        table_name,
    )

    benchmark_row = (
        batch_load_ndjson_from_gcs(
            table_name=table_name,
            gcs_uri=staged_table["gcs_uri"],
            table_ref=staged_table["table_ref"],
            schema=staged_table["schema"],
            expected_row_count=len(
                staged_table["rows"]
            ),
            source_bytes=staged_table[
                "source_bytes"
            ],
        )
    )

    load_benchmark_rows.append(
        benchmark_row
    )

    print(
        "Loaded rows:",
        benchmark_row["row_count"],
    )

    print(
        "Load seconds:",
        benchmark_row["load_seconds"],
    )

print(
    "\nCompleted batch loads:",
    len(load_benchmark_rows),
)


Batch loading: generation_runs
Started load job: 4a48c88d-df81-4bf8-bc2c-3c2d0dea70d7 | table: generation_runs
Loaded rows: 1
Load seconds: 3.3802

Batch loading: synthetic_customers
Started load job: 60bfcdf6-e976-433c-9183-b29ed10def7d | table: synthetic_customers
Loaded rows: 120
Load seconds: 2.938

Batch loading: synthetic_products
Started load job: 6b890756-a93d-4bcd-8b93-6973cb8d2a31 | table: synthetic_products
Loaded rows: 30
Load seconds: 2.3858

Batch loading: synthetic_orders
Started load job: 09cf262c-4825-4e8b-a589-96183963cf07 | table: synthetic_orders
Loaded rows: 400
Load seconds: 3.0401

Batch loading: synthetic_order_items
Started load job: 79bcf4c0-c17a-41c4-aa1b-584ab0c9af88 | table: synthetic_order_items
Loaded rows: 1193
Load seconds: 2.2608

Batch loading: synthetic_product_chunks
Started load job: 3b3f120f-e198-4291-a2e1-3208db2416ee | table: synthetic_product_chunks
Loaded rows: 174
Load seconds: 4.0254

Completed batch loads: 6


In [33]:
benchmark_blob_name = (
    f"{GCS_STAGING_PREFIX}/"
    f"{run_id}/"
    "batch_load_benchmarks.ndjson"
)

benchmark_gcs_uri, benchmark_source_bytes = (
    upload_rows_as_ndjson(
        load_benchmark_rows,
        blob_name=benchmark_blob_name,
    )
)

benchmark_job_config = bigquery.LoadJobConfig(
    schema=benchmark_schema,
    source_format=(
        bigquery.SourceFormat
        .NEWLINE_DELIMITED_JSON
    ),
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

benchmark_load_job = (
    bigquery_client.load_table_from_uri(
        benchmark_gcs_uri,
        benchmark_table_ref,
        location=dataset.location,
        job_config=benchmark_job_config,
    )
)

benchmark_load_job.result()

print(
    "Benchmark GCS URI:",
    benchmark_gcs_uri,
)

print(
    "Loaded benchmark rows:",
    len(load_benchmark_rows),
)

Benchmark GCS URI: gs://leafy-guide-497515-m4-vector-assets/synthetic-data-factory/staging/59eda7f6-2bce-4ecc-b3be-e93d6137c658/batch_load_benchmarks.ndjson
Loaded benchmark rows: 6


In [34]:
sql = f"""
SELECT
  (SELECT COUNT(*) FROM `{run_table_ref}`) AS run_count,
  (SELECT COUNT(*) FROM `{customer_table_ref}`) AS customer_count,
  (SELECT COUNT(*) FROM `{product_table_ref}`) AS product_count,
  (SELECT COUNT(*) FROM `{order_table_ref}`) AS order_count,
  (SELECT COUNT(*) FROM `{order_item_table_ref}`) AS order_item_count,
  (SELECT COUNT(*) FROM `{chunk_table_ref}`) AS chunk_count,
  (SELECT COUNT(*) FROM `{benchmark_table_ref}`) AS benchmark_count,
  (
    SELECT ARRAY_LENGTH(embedding)
    FROM `{chunk_table_ref}`
    LIMIT 1
  ) AS embedding_length
"""

verification_result = list(
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
)[0]

print(
    "Run count:",
    verification_result.run_count,
)

print(
    "Customer count:",
    verification_result.customer_count,
)

print(
    "Product count:",
    verification_result.product_count,
)

print(
    "Order count:",
    verification_result.order_count,
)

print(
    "Order item count:",
    verification_result.order_item_count,
)

print(
    "Chunk count:",
    verification_result.chunk_count,
)

print(
    "Benchmark count:",
    verification_result.benchmark_count,
)

print(
    "Embedding length:",
    verification_result.embedding_length,
)

Run count: 1
Customer count: 120
Product count: 30
Order count: 400
Order item count: 1193
Chunk count: 174
Benchmark count: 6
Embedding length: 768


In [35]:
sql = f"""
SELECT
  table_name,
  row_count,
  source_bytes,
  load_seconds,
  ROUND(
    SAFE_DIVIDE(row_count, load_seconds),
    2
  ) AS rows_per_second,
  bigquery_job_id
FROM `{benchmark_table_ref}`
ORDER BY load_seconds DESC
"""

batch_load_benchmark_df = (
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
    .to_dataframe()
)

batch_load_benchmark_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,row_count,source_bytes,load_seconds,rows_per_second,bigquery_job_id
0,synthetic_product_chunks,174,3120334,4.0254,43.23,3b3f120f-e198-4291-a2e1-3208db2416ee
1,generation_runs,1,2017,3.3802,0.30,4a48c88d-df81-4bf8-bc2c-3c2d0dea70d7
2,synthetic_orders,400,123091,3.0401,131.57,09cf262c-4825-4e8b-a589-96183963cf07
3,synthetic_customers,120,40204,2.9380,40.84,60bfcdf6-e976-433c-9183-b29ed10def7d
4,synthetic_products,30,85589,2.3858,12.57,6b890756-a93d-4bcd-8b93-6973cb8d2a31
5,synthetic_order_items,1193,325703,2.2608,527.69,79bcf4c0-c17a-41c4-aa1b-584ab0c9af88


In [36]:
def get_query_embedding(
    query: str,
) -> list[float]:
    embeddings = (
        text_embedding_model.get_embeddings(
            [query]
        )
    )

    return list(
        embeddings[0].values
    )


def search_products_semantically(
    query: str,
    *,
    top_k: int = 10,
) -> list[dict]:
    query_embedding = get_query_embedding(
        query
    )

    sql = f"""
    SELECT
      base.chunk_id,
      base.product_id,
      base.sku,
      base.category,
      base.title,
      base.chunk_number,
      base.chunk_text,
      distance
    FROM VECTOR_SEARCH(
      TABLE `{chunk_table_ref}`,
      'embedding',
      (
        SELECT
          @query_embedding AS embedding
      ),
      top_k => @top_k,
      distance_type => 'COSINE'
    )
    ORDER BY distance ASC
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "query_embedding",
                "FLOAT64",
                query_embedding,
            ),
            bigquery.ScalarQueryParameter(
                "top_k",
                "INT64",
                top_k,
            ),
        ]
    )

    query_job = bigquery_client.query(
        sql,
        job_config=job_config,
        location=dataset.location,
    )

    return [
        {
            "chunk_id": row.chunk_id,
            "product_id": row.product_id,
            "sku": row.sku,
            "category": row.category,
            "title": row.title,
            "chunk_number": row.chunk_number,
            "chunk_text": row.chunk_text,
            "distance": row.distance,
        }
        for row in query_job
    ]

In [37]:
semantic_query = """
Find portable and durable equipment for technical field teams
working outdoors or at remote infrastructure sites.
The equipment should help with inspection, diagnostics,
monitoring, or maintenance.
"""

semantic_results = (
    search_products_semantically(
        semantic_query,
        top_k=10,
    )
)

print("QUERY:")
print(semantic_query)

print(
    "\nSEMANTIC RESULTS:",
    len(semantic_results),
)

for result in semantic_results:
    print("=" * 100)

    print(
        "Distance:",
        round(result["distance"], 4),
    )

    print(
        "SKU:",
        result["sku"],
    )

    print(
        "Category:",
        result["category"],
    )

    print(
        "Title:",
        result["title"],
    )

    print(
        "Chunk:",
        result["chunk_number"],
    )

    print(
        result["chunk_text"][:900]
    )

QUERY:

Find portable and durable equipment for technical field teams
working outdoors or at remote infrastructure sites.
The equipment should help with inspection, diagnostics,
monitoring, or maintenance.


SEMANTIC RESULTS: 10
Distance: 0.1967
SKU: FIELD-0007
Category: Test & Measurement Instruments
Title: Rugged Field Test & Measurement Unit
Chunk: 5
remote locations; Preventative maintenance and troubleshooting of electrical and mechanical assets. Search concepts: Field testing equipment, Industrial measurement, Diagnostic instrument, Rugged tester, Site analysis tool, Maintenance diagnostics, Technical engineering, Portable data logger. Technical attributes: {"environment": "Harsh outdoor and industrial settings", "portability": "Highly portable with ergonomic design", "durability": "Extreme resistance to impact, dust, and moisture", "primary_user": "Field engineers, industrial technicians, maintenance specialists"}. Unit price: 378.97 USD. Available stock: 139 units.
Distance: 0.

In [38]:
def build_product_rag_context(
    results: list[dict],
) -> str:
    parts = []

    seen_chunk_ids = set()

    for index, result in enumerate(
        results,
        start=1,
    ):
        if result["chunk_id"] in seen_chunk_ids:
            continue

        seen_chunk_ids.add(
            result["chunk_id"]
        )

        parts.append(
            "\n".join(
                [
                    f"[SOURCE {index}]",
                    (
                        f"SKU: "
                        f"{result['sku']}"
                    ),
                    (
                        f"Category: "
                        f"{result['category']}"
                    ),
                    (
                        f"Product title: "
                        f"{result['title']}"
                    ),
                    (
                        f"Chunk number: "
                        f"{result['chunk_number']}"
                    ),
                    (
                        f"Distance: "
                        f"{result['distance']:.4f}"
                    ),
                    (
                        f"Chunk text: "
                        f"{result['chunk_text']}"
                    ),
                ]
            )
        )

    return "\n\n---\n\n".join(parts)

In [39]:
def answer_marketplace_question(
    question: str,
    *,
    top_k: int = 10,
) -> dict:
    results = (
        search_products_semantically(
            question,
            top_k=top_k,
        )
    )

    context = build_product_rag_context(
        results
    )

    prompt = f"""
You are a B2B technical marketplace assistant.

Answer using only the retrieved semantic product chunks.

Customer question:
{question}

Retrieved product chunks:
{context}

Return:
1. Short recommendation
2. Best matching products
3. Why each product is relevant
4. Likely use case
5. Important limitations or missing information
6. Sources used, referencing SOURCE numbers

Rules:
- Stay grounded in retrieved chunks.
- Do not invent product specifications.
- Do not claim a capability unless it appears in the context.
- Be practical and technical.
"""

    response = google_vertex_client.models.generate_content(
        model=TEXT_MODEL,
        contents=prompt,
    )

    return {
        "question": question,
        "answer": response.text,
        "search_results": results,
        "rag_context": context,
    }

In [40]:
marketplace_question = """
Our infrastructure team needs portable equipment for remote field work.

The engineers perform inspections, diagnose equipment problems,
and monitor technical systems in harsh outdoor environments.

Which products from the synthetic catalog are most relevant and why?
"""

marketplace_rag_result = (
    answer_marketplace_question(
        marketplace_question,
        top_k=10,
    )
)

print("CUSTOMER QUESTION:")
print(
    marketplace_rag_result["question"]
)

print("\nRAG ANSWER:")
print(
    marketplace_rag_result["answer"]
)

CUSTOMER QUESTION:

Our infrastructure team needs portable equipment for remote field work.

The engineers perform inspections, diagnose equipment problems,
and monitor technical systems in harsh outdoor environments.

Which products from the synthetic catalog are most relevant and why?


RAG ANSWER:
Your infrastructure team requires portable, rugged equipment for inspections, diagnostics, and monitoring in harsh outdoor environments. Several products from the catalog are highly relevant for these needs.

### 1. Short recommendation
Based on your team's requirements for portable equipment to perform inspections, diagnose problems, and monitor systems in harsh outdoor and remote environments, the **Rugged Field Test & Measurement Units** and **Rugged Field Diagnostic & Imaging Systems** are the most relevant categories.

### 2. Best matching products

*   **FIELD-0007**: Rugged Field Test & Measurement Unit
*   **FIELD-0025**: Rugged Multi-Parameter Field Diagnostic Instrument
*   **FIE

In [41]:
sql = f"""
WITH product_sales AS (
  SELECT
    p.product_id,
    p.sku,
    p.category,
    p.marketing_title,
    p.stock_quantity,
    COUNT(DISTINCT oi.order_id) AS order_count,
    SUM(oi.quantity) AS units_sold,
    ROUND(SUM(oi.line_total), 2) AS revenue
  FROM `{product_table_ref}` AS p
  LEFT JOIN `{order_item_table_ref}` AS oi
    ON p.product_id = oi.product_id
  GROUP BY
    p.product_id,
    p.sku,
    p.category,
    p.marketing_title,
    p.stock_quantity
)

SELECT
  category,
  COUNT(*) AS product_count,
  SUM(order_count) AS total_orders,
  SUM(units_sold) AS total_units_sold,
  ROUND(SUM(revenue), 2) AS total_revenue,
  ROUND(AVG(stock_quantity), 2) AS avg_stock_quantity
FROM product_sales
GROUP BY category
ORDER BY total_revenue DESC
"""

category_sales_df = (
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
    .to_dataframe()
)

category_sales_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category,product_count,total_orders,total_units_sold,total_revenue,avg_stock_quantity
0,Test & Measurement Instruments,5,189,687,336726.77,122.0
1,Power Tools & Welding Supplies,5,213,762,324805.36,196.8
2,Consumables & Industrial Chemicals,5,187,659,222127.52,126.2
3,Safety & Personal Protective Equipment (PPE),5,202,704,184976.13,150.8
4,Diagnostic & Imaging Systems,5,186,650,184180.53,179.4
5,Heavy Machinery & Lifting Gear,5,216,816,165031.42,138.6


In [42]:
sql = f"""
SELECT
  c.customer_segment,
  c.company_size,
  COUNT(DISTINCT c.customer_id) AS customer_count,
  COUNT(DISTINCT o.order_id) AS order_count,
  ROUND(SUM(o.total_amount), 2) AS total_order_value,
  ROUND(AVG(o.total_amount), 2) AS avg_order_value
FROM `{customer_table_ref}` AS c
LEFT JOIN `{order_table_ref}` AS o
  ON c.customer_id = o.customer_id
GROUP BY
  c.customer_segment,
  c.company_size
ORDER BY
  total_order_value DESC
"""

customer_segment_df = (
    bigquery_client.query(
        sql,
        location=dataset.location,
    )
    .to_dataframe()
)

customer_segment_df

/home/lipov/projects/Python_practice_sessions_05/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_segment,company_size,customer_count,order_count,total_order_value,avg_order_value
0,Telecommunications,Large Enterprise (501-5000 employees),6,28,120451.87,4301.85
1,Energy & Utilities,Medium Business (51-500 employees),12,38,115209.93,3031.84
2,Public Sector Services,Corporate Conglomerate (5001+ employees),8,32,107498.19,3359.32
3,Telecommunications,Small Business (1-50 employees),9,26,98868.84,3802.65
4,Civil Engineering & Construction,Medium Business (51-500 employees),7,23,84964.48,3694.11
5,Public Sector Services,Large Enterprise (501-5000 employees),9,24,84623.35,3525.97
6,Public Sector Services,Medium Business (51-500 employees),5,26,82170.88,3160.42
7,Manufacturing & Processing,Medium Business (51-500 employees),6,21,71472.32,3403.44
8,Public Sector Services,Small Business (1-50 employees),5,19,70397.81,3705.15
9,Civil Engineering & Construction,Corporate Conglomerate (5001+ employees),6,15,68914.07,4594.27


In [43]:
factory_report = {
    "run_id": run_id,
    "created_at": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    "notebook": (
        "65_w_google_cloud_"
        "synthetic_data_factory_batch_load.ipynb"
    ),
    "marketplace_plan": marketplace_plan,
    "generation": {
        "customer_count": len(customer_rows),
        "product_count": len(product_rows),
        "order_count": len(order_rows),
        "order_item_count": len(order_item_rows),
        "chunk_count": len(chunk_rows),
    },
    "performance": {
        "async_product_enrichment_seconds": (
            round(enrichment_seconds, 2)
        ),
        "embedding_seconds": round(
            embedding_seconds,
            2,
        ),
        "batch_load_benchmarks": (
            batch_load_benchmark_df
            .to_dict(orient="records")
        ),
    },
    "integrity": {
        "invalid_order_customer_refs": (
            len(invalid_order_customers)
        ),
        "invalid_item_order_refs": (
            len(invalid_item_orders)
        ),
        "invalid_item_product_refs": (
            len(invalid_item_products)
        ),
        "order_total_mismatches": (
            len(order_total_mismatches)
        ),
    },
    "semantic_search": {
        "query": semantic_query,
        "result_count": len(
            semantic_results
        ),
    },
    "rag_result": marketplace_rag_result,
    "category_sales": (
        category_sales_df
        .to_dict(orient="records")
    ),
    "customer_segments": (
        customer_segment_df
        .to_dict(orient="records")
    ),
    "local_files_saved": False,
}

factory_report_json = json.dumps(
    factory_report,
    indent=2,
    ensure_ascii=False,
    default=str,
)

report_blob_name = (
    f"{GCS_REPORT_PREFIX}/"
    f"{run_id}/"
    "factory_report.json"
)

report_gcs_uri, report_bytes = (
    upload_text_to_gcs(
        factory_report_json,
        blob_name=report_blob_name,
        content_type="application/json",
    )
)

print("Factory report saved to:")
print(report_gcs_uri)

print(
    "Report bytes:",
    report_bytes,
)

Factory report saved to:
gs://leafy-guide-497515-m4-vector-assets/synthetic-data-factory/reports/59eda7f6-2bce-4ecc-b3be-e93d6137c658/factory_report.json
Report bytes: 33820


In [44]:
notebook_summary = {
    "project_id": PROJECT_ID,
    "vertex_ai_location": LOCATION,
    "bigquery_location": dataset.location,
    "bucket_location": BUCKET_LOCATION,
    "created_at": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    "notebook": (
        "65_w_google_cloud_"
        "synthetic_data_factory_batch_load.ipynb"
    ),
    "run_id": run_id,
    "models": {
        "planning_model": PLANNING_MODEL,
        "text_model": TEXT_MODEL,
        "text_embedding_model": (
            TEXT_EMBEDDING_MODEL
        ),
    },
    "bigquery": {
        "dataset": DATASET_ID,
        "run_table": run_table_ref,
        "customer_table": customer_table_ref,
        "product_table": product_table_ref,
        "order_table": order_table_ref,
        "order_item_table": order_item_table_ref,
        "chunk_table": chunk_table_ref,
        "benchmark_table": benchmark_table_ref,
    },
    "cloud_storage": {
        "staging_prefix": (
            f"gs://{BUCKET_NAME}/"
            f"{GCS_STAGING_PREFIX}/{run_id}"
        ),
        "report_gcs_uri": report_gcs_uri,
        "local_files_saved": False,
    },
    "counts": {
        "customers": len(customer_rows),
        "products": len(product_rows),
        "orders": len(order_rows),
        "order_items": len(order_item_rows),
        "chunks": len(chunk_rows),
    },
    "performance": {
        "product_enrichment_seconds": (
            round(enrichment_seconds, 2)
        ),
        "embedding_seconds": round(
            embedding_seconds,
            2,
        ),
    },
}

summary_json = json.dumps(
    notebook_summary,
    indent=2,
    ensure_ascii=False,
    default=str,
)

summary_blob_name = (
    f"{GCS_SUMMARY_PREFIX}/"
    f"{run_id}/"
    "summary.json"
)

summary_gcs_uri, summary_bytes = (
    upload_text_to_gcs(
        summary_json,
        blob_name=summary_blob_name,
        content_type="application/json",
    )
)

print("Notebook summary saved to:")
print(summary_gcs_uri)

print(
    "Summary bytes:",
    summary_bytes,
)

Notebook summary saved to:
gs://leafy-guide-497515-m4-vector-assets/synthetic-data-factory/summaries/59eda7f6-2bce-4ecc-b3be-e93d6137c658/summary.json
Summary bytes: 1806


In [45]:
print(
    "Synthetic Marketplace Data Factory completed."
)

print("=" * 100)

print("Run ID:", run_id)

print(
    "Marketplace:",
    marketplace_plan["marketplace_name"],
)

print("\nGenerated relational data:")

print(
    "Customers:",
    len(customer_rows),
)

print(
    "Products:",
    len(product_rows),
)

print(
    "Orders:",
    len(order_rows),
)

print(
    "Order items:",
    len(order_item_rows),
)

print(
    "Semantic chunks:",
    len(chunk_rows),
)

print("\nPerformance:")

print(
    "Async Gemini enrichment:",
    round(enrichment_seconds, 2),
    "seconds",
)

print(
    "Embedding generation:",
    round(embedding_seconds, 2),
    "seconds",
)

print("\nBigQuery batch load benchmarks:")

for row in (
    batch_load_benchmark_df
    .to_dict(orient="records")
):
    print("=" * 100)

    print(
        "Table:",
        row["table_name"],
    )

    print(
        "Rows:",
        row["row_count"],
    )

    print(
        "Source bytes:",
        row["source_bytes"],
    )

    print(
        "Load seconds:",
        row["load_seconds"],
    )

    print(
        "Rows per second:",
        row["rows_per_second"],
    )

print("\nCloud outputs:")

print(
    "Staging prefix:",
    (
        f"gs://{BUCKET_NAME}/"
        f"{GCS_STAGING_PREFIX}/{run_id}"
    ),
)

print(
    "Factory report:",
    report_gcs_uri,
)

print(
    "Notebook summary:",
    summary_gcs_uri,
)

print("\nNo local files were saved.")

Synthetic Marketplace Data Factory completed.
Run ID: 59eda7f6-2bce-4ecc-b3be-e93d6137c658
Marketplace: InfraTech Exchange

Generated relational data:
Customers: 120
Products: 30
Orders: 400
Order items: 1193
Semantic chunks: 174

Performance:
Async Gemini enrichment: 124.32 seconds
Embedding generation: 7.53 seconds

BigQuery batch load benchmarks:
Table: synthetic_product_chunks
Rows: 174
Source bytes: 3120334
Load seconds: 4.0254
Rows per second: 43.23
Table: generation_runs
Rows: 1
Source bytes: 2017
Load seconds: 3.3802
Rows per second: 0.3
Table: synthetic_orders
Rows: 400
Source bytes: 123091
Load seconds: 3.0401
Rows per second: 131.57
Table: synthetic_customers
Rows: 120
Source bytes: 40204
Load seconds: 2.938
Rows per second: 40.84
Table: synthetic_products
Rows: 30
Source bytes: 85589
Load seconds: 2.3858
Rows per second: 12.57
Table: synthetic_order_items
Rows: 1193
Source bytes: 325703
Load seconds: 2.2608
Rows per second: 527.69

Cloud outputs:
Staging prefix: gs://leafy-